# Minimal GPT — Simplest Possible Implementation

A bare-bones, character-level GPT in ~150 lines of PyTorch.  
Same architecture as `gpt_from_scratch.ipynb`, zero boilerplate.

**Kept:** token embed → pos embed → N × (attention + FFN) → lm_head  
**Dropped:** all visualisations, verbose comments, attention-weight inspection

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import pandas as pd

torch.manual_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Hyperparameters (change only here) ───────────────────────────────────────
block_size    = 256
batch_size    = 32
n_embd        = 192
n_head        = 6
n_layer       = 4
dropout       = 0.2
learning_rate = 3e-4
max_iters     = 5000
eval_interval = 500
eval_iters    = 200

print(f'device: {device}')

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────────
df   = pd.read_csv(r'medquad.csv\medquad.csv')
text = ''.join(
    f'Q: {str(r["question"]).strip()}\nA: {str(r["answer"]).strip()}\n\n'
    for _, r in df.iterrows()
)

# ── Tokenizer (character-level) ───────────────────────────────────────────────
chars      = sorted(set(text))
vocab_size = len(chars)
stoi       = {ch: i for i, ch in enumerate(chars)}
itos       = {i: ch for i, ch in enumerate(chars)}
encode     = lambda s: [stoi[c] for c in s]
decode     = lambda l: ''.join(itos[i] for i in l)

# ── Train / val split ─────────────────────────────────────────────────────────
data       = torch.tensor(encode(text), dtype=torch.long)
n          = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

print(f'vocab_size={vocab_size}  train={len(train_data):,}  val={len(val_data):,}')

In [ ]:
def get_batch(split):
    d  = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x  = torch.stack([d[i   : i + block_size    ] for i in ix])
    y  = torch.stack([d[i+1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.k = nn.Linear(n_embd, head_size, bias=False)
        self.q = nn.Linear(n_embd, head_size, bias=False)
        self.v = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        wei = self.q(x) @ self.k(x).transpose(-2, -1) * C**-0.5  # (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = self.drop(F.softmax(wei, dim=-1))
        return wei @ self.v(x)                                     # (B,T,hs)


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(num_heads * head_size, n_embd)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))


class FFN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.sa  = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffn = FFN()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks  = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T  = idx.shape
        x     = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=device))
        x     = self.ln_f(self.blocks(x))
        logits = self.lm_head(x)           # (B, T, vocab_size)
        loss   = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            probs     = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            idx       = torch.cat((idx, torch.multinomial(probs, 1)), dim=1)
        return idx


model = GPT().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ('train', 'val'):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


opt = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in range(max_iters):
    if step % eval_interval == 0:
        l = estimate_loss()
        print(f'step {step:5d}  train {l["train"]:.4f}  val {l["val"]:.4f}')
    x, y = get_batch('train')
    _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

print('Done')

In [ ]:
model.eval()
prompt = 'Q: What is diabetes?\nA:'
ctx    = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
out    = model.generate(ctx, max_new_tokens=300, temperature=0.8)
print(decode(out[0].tolist()))

In [ ]:
torch.save({'state': model.state_dict(), 'stoi': stoi, 'itos': itos}, 'gpt_minimal.pt')
print('Saved -> gpt_minimal.pt')